# Week 20: Experiment tracking with SQL, a demo app, and buffer time

A lighter week, by design. Use the spare hours to **catch up** on anything from Weeks 17–19. The two new skills here are among the most requested in AI jobs: **SQL** and **showing your work** with a demo.

| Session | Topic | Time |
|---|---|---|
| Day 1 | SQL basics with SQLite (built into Python) | 2 h |
| Day 2 | An experiment tracker: every run logged with its config, metrics and Git commit | 2.5 h |
| Day 3 | *(optional)* A Streamlit demo: search + image tab, with a disclaimer | 2.5 h |
| Buffer | Catch up on Weeks 17–19 | 3 h |

### Done when
- [ ] You can write SELECT / WHERE / GROUP BY / ORDER BY / JOIN queries without notes.
- [ ] `medlab/tracking.py` logs every CLI run to `runs.db`, and you can query "best run per system".
- [ ] *(optional)* `streamlit run app.py` shows a working demo.

In [ ]:
import sys, importlib, subprocess
from pathlib import Path
for p in ["..", "../capstone"]:
    if str(Path(p).resolve()) not in sys.path:
        sys.path.insert(0, str(Path(p).resolve()))
CAP = Path("../capstone")

def run_tests(*paths):
    """Run pytest inside the capstone folder and return its exit code (0 = all passed)."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *paths], cwd=CAP, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:])
    return r.returncode

def fresh(module_name):
    """Import (or re-import after you edited it) a module from your capstone package."""
    importlib.invalidate_caches()          # notice files you created after the kernel started
    return importlib.reload(importlib.import_module(module_name))

---
# Day 1: SQL in 2 hours (2 h)

Almost every company keeps its data in a SQL database. **SQLite** is a whole database in a single file, with no server, and Python ships with it (`import sqlite3`). The core of SQL:
```sql
CREATE TABLE runs (id INTEGER PRIMARY KEY, system TEXT, ndcg REAL);
INSERT INTO runs (system, ndcg) VALUES ('bm25', 0.32);
SELECT system, AVG(ndcg) AS mean_ndcg FROM runs WHERE ndcg > 0.1 GROUP BY system ORDER BY mean_ndcg DESC;
SELECT r.system, c.k1 FROM runs r JOIN configs c ON r.config_id = c.id;
```
**Always use `?` placeholders** for values. Never build SQL with f-strings: that's how SQL injection happens.

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")            # a throwaway database in RAM
con.executescript("""
CREATE TABLE patients_synthetic (id INTEGER PRIMARY KEY, age INTEGER, site TEXT);
CREATE TABLE scans (id INTEGER PRIMARY KEY, patient_id INTEGER, label INTEGER, model_score REAL,
                    FOREIGN KEY (patient_id) REFERENCES patients_synthetic(id));
""")
rows_p = [(1, 34, "A"), (2, 61, "A"), (3, 45, "B"), (4, 70, "B")]
rows_s = [(1, 1, 0, 0.2), (2, 1, 0, 0.3), (3, 2, 1, 0.9), (4, 3, 1, 0.6), (5, 4, 0, 0.7), (6, 4, 1, 0.8)]
con.executemany("INSERT INTO patients_synthetic VALUES (?, ?, ?)", rows_p)
con.executemany("INSERT INTO scans VALUES (?, ?, ?, ?)", rows_s)
print(con.execute("SELECT site, COUNT(*), AVG(s.model_score) FROM scans s JOIN patients_synthetic p ON s.patient_id = p.id GROUP BY site").fetchall())

### Exercise 20.1: Write the queries
Fill in each string so that the test passes. (These data are synthetic.)

In [ ]:
q1 = ""   # number of scans per patient, as (patient_id, n_scans), ordered by patient_id
q2 = ""   # ids of patients older than 50, ascending
q3 = ""   # the site with the highest mean model_score among POSITIVE scans (label = 1): one row (site, mean_score)
q4 = ""   # patients with more than one scan: (patient_id, age), ordered by patient_id

In [ ]:
assert con.execute(q1).fetchall() == [(1, 2), (2, 1), (3, 1), (4, 2)]
assert [r[0] for r in con.execute(q2).fetchall()] == [2, 4]
site, score = con.execute(q3).fetchall()[0]
assert site == "A" and abs(score - 0.9) < 1e-9
assert con.execute(q4).fetchall() == [(1, 34), (4, 70)]
print("✅ 20.1 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

q1: `SELECT patient_id, COUNT(*) FROM scans GROUP BY patient_id ORDER BY patient_id`.

</details>

<details><summary>Hint 2</summary>

q3 needs a JOIN, `WHERE s.label = 1`, `GROUP BY site`, `ORDER BY ... DESC LIMIT 1`. q4 needs `HAVING COUNT(*) > 1`.

</details>


---
# Day 2: A tiny experiment tracker (2.5 h)

Tools like MLflow and Weights & Biases do this at scale. Building a minimal version yourself teaches you *what* they store, and why: **every result must be traceable to the exact code (Git commit) and config that produced it.**

### Exercise 20.2: `capstone/medlab/tracking.py`
```python
init_db(path="runs.db")                                   # creates table runs(id, timestamp, experiment, name, git_commit, config_json, metrics_json)
current_git_commit() -> str                               # `git rev-parse --short HEAD` via subprocess; "unknown" if it fails
log_run(experiment, name, config: dict, metrics: dict, path="runs.db") -> int   # returns the new row id
best_runs(experiment, metric, path="runs.db") -> list[tuple[name, best_value]]  # the best value per name, highest first
```
Use `json.dumps` for the dicts and `json_extract(metrics_json, '$."ndcg@10"')` in SQL (SQLite has JSON functions built in). Then call `log_run` at the end of `medlab/evaluate.py` and `medlab/imaging/run.py`. Write `tests/test_tracking.py` using `tmp_path`.

In [ ]:
# Implement tracking.py and tests, hook it into both CLIs, then run the checker.

In [ ]:
assert run_tests("tests/test_tracking.py") == 0
tracking = fresh("medlab.tracking")
db = CAP / "_check_runs.db"
if db.exists(): db.unlink()
tracking.init_db(db)
tracking.log_run("retrieval", "bm25", {"k1": 1.2}, {"ndcg@10": 0.30}, db)
tracking.log_run("retrieval", "bm25", {"k1": 0.9}, {"ndcg@10": 0.32}, db)
rid = tracking.log_run("retrieval", "dense", {}, {"ndcg@10": 0.31}, db)
assert isinstance(rid, int)
best = tracking.best_runs("retrieval", "ndcg@10", db)
assert [b[0] for b in best] == ["bm25", "dense"] and abs(best[0][1] - 0.32) < 1e-12
db.unlink()
assert "log_run" in (CAP / "medlab" / "evaluate.py").read_text()
print("✅ 20.2 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

The query for `best_runs`: `SELECT name, MAX(json_extract(metrics_json, ?)) AS best FROM runs WHERE experiment = ? GROUP BY name ORDER BY best DESC`, with the parameter `'$."' + metric + '"'` (quotes are needed because of the `@`).

</details>


---
# Day 3 *(optional)*: A Streamlit demo (2.5 h)

`pip install streamlit`, then create `capstone/app.py`:
```python
import streamlit as st
st.set_page_config(page_title="MedSearch Lab")
st.warning("Research demo only. Not clinically validated. Do not use for medical decisions.")
tab1, tab2 = st.tabs(["Literature search", "Image model"])
with tab1:
    q = st.text_input("Query", "vitamin D and bone health")
    # load your index + BM25 once with @st.cache_resource, then show the top 10 titles and scores
with tab2:
    # let the user pick a TEST image by index (no uploads: avoids anyone uploading real patient data),
    # show it, the model's probability, and the true label
```
Run it with `streamlit run app.py`. Record a 60-second screen capture for your README; recruiters watch videos more often than they read code.

**Why no image uploads?** Inviting users to upload medical images creates privacy and safety obligations that a student demo can't meet. That design decision is worth mentioning in an interview.

---
# Buffer and review
Use the remaining time to finish Weeks 17–19. Then:
1. Run *every* CLI once more. Is each run in `runs.db`, with a commit hash?
2. *(Week 9)* What's in `DATASETS.md`? Is it up to date with PneumoniaMNIST?
3. *(Week 17)* Is there any deviation from `PROTOCOL.md` that you haven't written down yet?
4. Update the tracker, write the Month 5 summary in `LOG.md`, and **send me** the repo link.

✍️ *Your answer:* (double-click to edit)

